# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya - Exploration with `mlcroissant`

This notebook provides a step-by-step workflow for loading and exploring the [FAIR^2](https://doi.org/10.71728/senscience.y7m0-f273) dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema located at the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
We start by loading the dataset metadata and available records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the Dataset (schema and structure)
dataset = mlc.Dataset(croissant_url)

# Access the top-level metadata object
md = dataset.metadata
print(f"{md.name}: {md.description}")
print(f"License: {md.license}")
print(f"Authors: {[author['@id'] for author in getattr(md, 'author', [])]}")

## 2. Data Overview
List all available record sets, their `@id`s, and their fields/columns (referenced by `@id`).

In [ ]:
# Helper: List all record sets and their columns/fields by @id
record_sets = []
print("Available RecordSets:")
for record_set in dataset.record_sets:
    print(f"- RecordSet name: {record_set.name}")
    print(f"  @id: {record_set['@id']}")
    record_sets.append(record_set['@id'])
    # Columns (for tabular data)
    if hasattr(record_set, 'columns') and record_set.columns:
        print("  Columns (@id):")
        for col in record_set.columns:
            print(f"    - {col['@id']} (name: {col.name})")
    # Fields (for JSON/semistructured)
    if hasattr(record_set, 'fields') and record_set.fields:
        print("  Fields (@id):")
        for field in record_set.fields:
            print(f"    - {field['@id']} (name: {field.name})")
    print()

## 3. Data Extraction
Extract the data from each record set into a pandas DataFrame for further analysis. All record sets and fields referenced by their `@id` as above.

In [ ]:
# Prepare DataFrames from the record sets by their @id
dataframes = {}
# Confirm which record sets are found
print("Record sets to extract:")
for rid in record_sets:
    print(f"- {rid}")
    records = list(dataset.records(record_set=rid))
    if records:
        df = pd.DataFrame(records)
        dataframes[rid] = df
        print(f"  Loaded with {len(df)} rows and {len(df.columns)} columns.")
    else:
        print("  No records found or unable to load.")
print()
# Display quick preview for the first loaded record set
if dataframes:
    main_record_set_id = next(iter(dataframes))
    df = dataframes[main_record_set_id]
    print(f"First record set: {main_record_set_id}")
    print("Columns:", df.columns.tolist())
    display(df.head())
else:
    print("No dataframes loaded from record sets.")

## 4. Exploratory Data Analysis (EDA)

Apply typical data processing steps such as:
- Filtering records based on numeric field value
- Normalizing a numeric field
- Grouping data by a key attribute

All columns and fields are referenced strictly by their `@id` as discovered above.

In [ ]:
# ----
# For this notebook, we pick the first loaded record set and search for a numeric column
import numpy as np

if dataframes:
    # Get first loaded record set
    record_set_id = main_record_set_id
    df = dataframes[record_set_id]
    # Try to find a numeric column; otherwise, skip
    numeric_field_id = None
    for col in df.columns:
        # Quick check for numeric nature
        if np.issubdtype(df[col].dropna().dtype, np.number):
            numeric_field_id = col
            break
    if numeric_field_id is not None:
        print(f"Using numeric field '@id': {numeric_field_id}")
        threshold = df[numeric_field_id].mean() # Arbitrary, use mean as threshold
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Try to group by a categorical column
        group_field_id = None
        for col in df.columns:
            if col != numeric_field_id and df[col].dtype == object:
                group_field_id = col
                break
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Mean of {numeric_field_id} grouped by {group_field_id}:")
            display(grouped_df.head())
    else:
        print("No numeric field found in the record set for EDA.")
else:
    print("No data loaded for EDA.")

## 5. Visualization
Visualize numeric data distributions or relationships between columns. All references use `@id`s.

In [ ]:
# Visualization example: histogram and group bar plot
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field_id is not None:
    fig, axs = plt.subplots(1, 2, figsize=(12, 4))
    # Histogram of numeric field
    sns.histplot(df[numeric_field_id].dropna(), ax=axs[0], kde=True)
    axs[0].set_title(f"Distribution of {numeric_field_id}")
    # Bar plot: group by category if group_field_id available
    if group_field_id:
        sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id, ax=axs[1])
        axs[1].set_title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.setp(axs[1].get_xticklabels(), rotation=45, ha="right")
    else:
        axs[1].text(0.5, 0.5, "No grouping field", ha='center', va='center')
    plt.tight_layout()
    plt.show()
else:
    print("No numeric field found for visualization.")

## 6. Conclusion

In this notebook, we demonstrated how to:

- Load and review the metadata and structure of a dataset defined by the Croissant schema and accessible using the `mlcroissant` library
- Identify and extract data from available record sets by their `@id`
- Perform basic exploratory data analysis and normalization on numeric fields
- Visualize data distributions and aggregated measures, all while referencing dataset elements by their `@id`

This workflow provides a foundation for further detailed exploration and statistical modeling of the predictors and outcomes described in the FAIR^2 dataset.